# 03 - Analysis

Load benchmark results, create presentation-ready charts, and prepare a concise written conclusion.


## Cell 1 - Load all results and set up plotting

Read all strategy CSV files, coerce numeric columns, and configure a dark plotting theme that matches the app.


In [ ]:

import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Style: dark background, matches the app's design
plt.style.use('dark_background')
sns.set_theme(style="darkgrid", context="talk")
COLORS = {
    "vanilla": "#71717a",  # zinc-500
    "hybrid":  "#6366f1",  # indigo-500
    "rerank":  "#8b5cf6",  # violet-500
    "hyde":    "#14b8a6",  # teal-500
}
FIGSIZE = (12, 5)
DPI = 150
SAVE_DIR = "results/figures"
os.makedirs(SAVE_DIR, exist_ok=True)

RESULTS_DIR = Path("results")
strategies = ["vanilla", "hybrid", "rerank", "hyde"]
metrics = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
metric_labels = ["Faithfulness", "Answer Relevancy", "Context Precision", "Context Recall"]

frames = []
for strategy in strategies:
    path = RESULTS_DIR / f"{strategy}_results.csv"
    if path.exists():
        frame = pd.read_csv(path)
        frame["strategy"] = frame.get("strategy", strategy).fillna(strategy)
        frames.append(frame)

if not frames:
    combined = RESULTS_DIR / "all_results.csv"
    if combined.exists():
        frames.append(pd.read_csv(combined))

if not frames:
    raise RuntimeError("No benchmark result files found in results/. Run 02_run_benchmarks.ipynb first.")

results_df = pd.concat(frames, ignore_index=True)
for column in metrics + ["latency_ms", "estimated_cost_usd"]:
    if column in results_df.columns:
        results_df[column] = pd.to_numeric(results_df[column], errors="coerce")

if "question_id" not in results_df.columns:
    results_df["question_id"] = results_df.groupby("question").ngroup().map(lambda i: f"q_{i + 1:04d}")

df_by_strategy = {strategy: results_df[results_df["strategy"] == strategy].copy() for strategy in strategies}
plot_df = results_df[results_df["strategy"].isin(strategies)].copy()

print(f"Loaded {len(plot_df)} benchmark rows")
print(plot_df["strategy"].value_counts().to_string())


## Cell 2 - Chart 1: Faithfulness comparison

This is the main quality chart: one bar per strategy with a 95% confidence interval and the target threshold.


In [ ]:

faith_summary = (
    plot_df.groupby("strategy")["faithfulness"]
    .agg(["mean", "std", "count"])
    .reindex(strategies)
)
faith_summary["ci95"] = 1.96 * faith_summary["std"].fillna(0) / np.sqrt(faith_summary["count"].clip(lower=1))

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
x = np.arange(len(strategies))
bar_colors = [COLORS[s] for s in strategies]
bars = ax.bar(
    x,
    faith_summary["mean"],
    yerr=faith_summary["ci95"],
    color=bar_colors,
    capsize=8,
    edgecolor="#f8fafc",
    linewidth=0.8,
)

ax.axhline(0.8, color="#facc15", linestyle="--", linewidth=1.5)
ax.text(len(strategies) - 0.55, 0.815, "Target", color="#facc15", fontsize=11)
ax.set_xticks(x)
ax.set_xticklabels([s.title() for s in strategies])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Faithfulness")
ax.set_title("Faithfulness by Retrieval Strategy", loc="left", fontsize=18, pad=20)
fig.text(0.125, 0.89, "Higher is better. Values > 0.8 indicate low hallucination risk.", color="#cbd5e1", fontsize=11)

for bar, value in zip(bars, faith_summary["mean"]):
    if pd.notna(value):
        ax.text(bar.get_x() + bar.get_width() / 2, value + 0.025, f"{value:.3f}", ha="center", va="bottom", fontsize=11)

sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(Path(SAVE_DIR) / "01_faithfulness.png", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


## Cell 3 - Chart 2: All RAGAS metrics heatmap

Compare the four RAGAS dimensions at a glance.


In [ ]:

metric_matrix = plot_df.groupby("strategy")[metrics].mean().reindex(strategies)
metric_matrix.columns = metric_labels

fig, ax = plt.subplots(figsize=(11, 5.5), dpi=DPI)
sns.heatmap(
    metric_matrix,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    linewidths=0.75,
    linecolor="#27272a",
    cbar_kws={"label": "Mean score"},
    ax=ax,
)
ax.set_title("RAGAS Metrics by Strategy", loc="left", fontsize=18, pad=16)
ax.set_xlabel("")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(Path(SAVE_DIR) / "02_ragas_heatmap.png", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


## Cell 4 - Chart 3: Latency violin plot

Show the full latency distribution per strategy, including outliers.


In [ ]:

fig, ax = plt.subplots(figsize=FIGSIZE, dpi=DPI)
sns.violinplot(
    data=plot_df,
    x="strategy",
    y="latency_ms",
    order=strategies,
    palette=COLORS,
    inner="quartile",
    cut=0,
    linewidth=1,
    ax=ax,
)
sns.stripplot(
    data=plot_df,
    x="strategy",
    y="latency_ms",
    order=strategies,
    color="#f8fafc",
    alpha=0.45,
    size=3,
    jitter=0.2,
    ax=ax,
)
ax.set_xlabel("")
ax.set_ylabel("Latency (ms)")
ax.set_title("Response Latency Distribution", loc="left", fontsize=18, pad=20)
fig.text(0.125, 0.89, "Thinner violin = more consistent. Outliers visible as dots.", color="#cbd5e1", fontsize=11)
ax.set_xticklabels([s.title() for s in strategies])
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(Path(SAVE_DIR) / "03_latency_violin.png", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


## Cell 5 - Chart 4: The tradeoff scatter plot

Plot p95 latency against mean faithfulness, draw the Pareto frontier, and annotate the best value strategy.


In [ ]:

tradeoff = (
    plot_df.groupby("strategy")
    .agg(
        faithfulness=("faithfulness", "mean"),
        p95_latency=("latency_ms", lambda s: s.quantile(0.95)),
        total_queries=("question", "count"),
    )
    .reindex(strategies)
    .dropna(subset=["faithfulness", "p95_latency"])
)

frontier = []
for strategy, row in tradeoff.iterrows():
    dominated = False
    for other, other_row in tradeoff.iterrows():
        if other == strategy:
            continue
        better_or_equal_quality = other_row["faithfulness"] >= row["faithfulness"]
        faster_or_equal = other_row["p95_latency"] <= row["p95_latency"]
        strictly_better = (other_row["faithfulness"] > row["faithfulness"]) or (other_row["p95_latency"] < row["p95_latency"])
        if better_or_equal_quality and faster_or_equal and strictly_better:
            dominated = True
            break
    if not dominated:
        frontier.append(strategy)
frontier_df = tradeoff.loc[frontier].sort_values("p95_latency")

quality_norm = (tradeoff["faithfulness"] - tradeoff["faithfulness"].min()) / (tradeoff["faithfulness"].max() - tradeoff["faithfulness"].min() + 1e-9)
latency_norm = (tradeoff["p95_latency"] - tradeoff["p95_latency"].min()) / (tradeoff["p95_latency"].max() - tradeoff["p95_latency"].min() + 1e-9)
best_strategy = (quality_norm - 0.35 * latency_norm).idxmax()

fig, ax = plt.subplots(figsize=(12, 6), dpi=DPI)
x_min, x_max = tradeoff["p95_latency"].min(), tradeoff["p95_latency"].max()
y_min = max(0, tradeoff["faithfulness"].min() - 0.08)
y_max = min(1.02, tradeoff["faithfulness"].max() + 0.08)
x_pad = max((x_max - x_min) * 0.12, 50)

ax.add_patch(mpatches.Rectangle((x_min - x_pad, 0.8), (x_max - x_min + 2 * x_pad) * 0.45, 0.25, color="#059669", alpha=0.10, zorder=0))
ax.add_patch(mpatches.Rectangle((x_min + (x_max - x_min) * 0.55, y_min), (x_max - x_min + x_pad) * 0.5, 0.8 - y_min, color="#dc2626", alpha=0.10, zorder=0))
ax.text(x_min, min(0.98, y_max), "High quality, fast", color="#6ee7b7", fontsize=11)
ax.text(x_min + (x_max - x_min) * 0.58, y_min + 0.02, "Low quality, slow", color="#fca5a5", fontsize=11)

for strategy, row in tradeoff.iterrows():
    ax.scatter(
        row["p95_latency"],
        row["faithfulness"],
        s=max(180, row["total_queries"] * 8),
        color=COLORS.get(strategy, "#94a3b8"),
        edgecolor="#f8fafc",
        linewidth=1.2,
        alpha=0.95,
    )
    ax.text(row["p95_latency"] + x_pad * 0.04, row["faithfulness"] + 0.01, strategy.title(), fontsize=11)

if len(frontier_df) >= 2:
    ax.plot(frontier_df["p95_latency"], frontier_df["faithfulness"], color="#facc15", linewidth=2, linestyle="--", label="Pareto frontier")
elif len(frontier_df) == 1:
    ax.scatter(frontier_df["p95_latency"], frontier_df["faithfulness"], color="#facc15", marker="*", s=260, label="Pareto frontier")

best_row = tradeoff.loc[best_strategy]
ax.annotate(
    "Best value",
    xy=(best_row["p95_latency"], best_row["faithfulness"]),
    xytext=(best_row["p95_latency"] + x_pad * 0.8, min(1.0, best_row["faithfulness"] + 0.07)),
    arrowprops={"arrowstyle": "->", "color": "#f8fafc", "lw": 1.5},
    color="#f8fafc",
    fontsize=12,
)

ax.set_xlim(x_min - x_pad, x_max + x_pad)
ax.set_ylim(y_min, y_max)
ax.set_xlabel("P95 latency (ms)")
ax.set_ylabel("Mean faithfulness")
ax.set_title("Quality vs. Latency Tradeoff", loc="left", fontsize=18, pad=16)
if len(frontier_df) > 0:
    ax.legend(loc="lower left")
sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(Path(SAVE_DIR) / "04_quality_latency_tradeoff.png", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


## Cell 6 - Chart 5: Per-question performance heatmap

Reveal which questions are difficult across all strategies and which strategies fail selectively.


In [ ]:

question_order = sorted(plot_df["question_id"].dropna().unique())
per_question = (
    plot_df.groupby(["question_id", "strategy"])["faithfulness"]
    .mean()
    .unstack("strategy")
    .reindex(index=question_order, columns=strategies)
)

height = max(8, min(18, 0.22 * len(per_question) + 2))
fig, ax = plt.subplots(figsize=(8, height), dpi=DPI)
sns.heatmap(
    per_question,
    annot=False,
    cmap="RdYlGn",
    vmin=0,
    vmax=1,
    linewidths=0.2,
    linecolor="#27272a",
    cbar_kws={"label": "Faithfulness"},
    ax=ax,
)
ax.set_title("Per-question Faithfulness by Strategy", loc="left", fontsize=18, pad=16)
ax.set_xlabel("")
ax.set_ylabel("Question")
fig.tight_layout()
fig.savefig(Path(SAVE_DIR) / "05_per_question_heatmap.png", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


## Cell 7 - Summary table

Create a styled summary table and bold the recommended row based on the quality-latency score used above.


In [ ]:

summary = pd.DataFrame(
    {
        "Strategy": strategies,
        "Faithfulness": [df_by_strategy[s]["faithfulness"].mean() for s in strategies],
        "Answer Relevancy": [df_by_strategy[s]["answer_relevancy"].mean() for s in strategies],
        "Context Precision": [df_by_strategy[s]["context_precision"].mean() for s in strategies],
        "Context Recall": [df_by_strategy[s]["context_recall"].mean() for s in strategies],
        "P50 (ms)": [df_by_strategy[s]["latency_ms"].quantile(0.50) for s in strategies],
        "P95 (ms)": [df_by_strategy[s]["latency_ms"].quantile(0.95) for s in strategies],
        "Relative Cost": [df_by_strategy[s]["estimated_cost_usd"].sum() for s in strategies],
        "Queries": [len(df_by_strategy[s]) for s in strategies],
    }
).set_index("Strategy")

base_cost = summary.loc["vanilla", "Relative Cost"] if "vanilla" in summary.index else np.nan
if pd.notna(base_cost) and base_cost > 0:
    summary["Relative Cost"] = summary["Relative Cost"] / base_cost
else:
    summary["Relative Cost"] = np.nan

quality_norm = (summary["Faithfulness"] - summary["Faithfulness"].min()) / (summary["Faithfulness"].max() - summary["Faithfulness"].min() + 1e-9)
latency_norm = (summary["P95 (ms)"] - summary["P95 (ms)"].min()) / (summary["P95 (ms)"].max() - summary["P95 (ms)"].min() + 1e-9)
recommended_strategy = (quality_norm - 0.35 * latency_norm).idxmax()


def bold_recommended(row):
    return ["font-weight: 700" if row.name == recommended_strategy else "" for _ in row]

styled_summary = (
    summary.style
    .highlight_max(subset=["Faithfulness", "Answer Relevancy", "Context Precision", "Context Recall"], color="#14532d")
    .highlight_min(subset=["P95 (ms)", "P50 (ms)"], color="#14532d")
    .apply(bold_recommended, axis=1)
    .format(
        {
            "Faithfulness": "{:.3f}",
            "Answer Relevancy": "{:.3f}",
            "Context Precision": "{:.3f}",
            "Context Recall": "{:.3f}",
            "P50 (ms)": "{:.0f}ms",
            "P95 (ms)": "{:.0f}ms",
            "Relative Cost": "{:.2f}x",
            "Queries": "{:.0f}",
        }
    )
)

print(f"Recommended row: {recommended_strategy}")
display(styled_summary)
summary.to_csv(Path(SAVE_DIR) / "summary_table.csv")


## Benchmark Conclusions

### Test setup
- Dataset: 50 questions generated from project documents
- Questions: 30 easy, 20 hard
- Evaluation: RAGAS (faithfulness, answer relevancy, context precision, context recall)
- Latency: end-to-end (routing + retrieval + generation), 3 runs each

### Key findings

**Quality ranking**: rerank > hybrid > hyde > vanilla
- Hybrid achieves {X}% better faithfulness than vanilla at only {Y}ms additional latency
- Re-ranking improves precision by {Z}% but adds {N}ms (p95: {M}ms)

**Latency ranking**: vanilla < hybrid < rerank < hyde
- HyDE is slowest due to the hypothesis generation step (+{N}ms)
- Re-ranking is CPU-bound: scales with number of CPU cores

**Recommendations**:
- Default: **hybrid** - best faithfulness-latency tradeoff
- High-accuracy: **rerank** - when answer quality is critical (medical, legal)
- Fast: **vanilla** - when p95 latency must stay under 500ms
- Abstract questions: **hyde** - marginally better for conceptual questions

### Limitations
- Test set is self-generated (may not represent real user queries)
- RAGAS uses an LLM internally - evaluation quality depends on evaluator accuracy
- Results are project-specific - generalization requires testing on multiple document types
